Kappa in A-S has units of 1/dollar and must be estimated from fill probability curves as a function of quote distance. Without full LOB depth data, this estimation is not possible. We therefore set the MM spread equal to the prevailing best bid-ask spread, consistent with the infinite-horizon result that the optimal spread converges to a market-determined constant at high κ. Inventory control is implemented entirely through reservation price skew, which is the economically meaningful component of A-S on tick-constrained crypto venues.

In [10]:
import numpy as np
import pandas as pd

def compute_as_quotes(
    best_bid: float,
    best_ask: float,
    mid: float,
    sigma: float,       # rolling vol, in return space (e.g. 0.000079)
    inventory: float,   # current inventory in base asset (e.g. BTC)
    gamma: float,       # risk aversion parameter
) -> tuple[float, float, float, float]:
    """
    Infinite-horizon Avellaneda-Stoikov quotes with inventory skew.

    Returns: (mm_bid, mm_ask, spread, skew)
    """

    # Step 1: Infinite-horizon A-S spread
    # Formula: (2/gamma) * ln(1 + gamma/kappa)
    as_spread = best_ask - best_bid

    # Step 2: Inventory skew
    # Formula: -q * gamma * sigma * S
    skew = -inventory * gamma * sigma * mid

    # Step 3: Reservation price (mid shifted by skew)
    reservation_price = mid + skew

    # Step 4: Place quotes symmetrically around reservation price
    # bid = reservation_price - half_spread
    # ask = reservation_price + half_spread
    half_spread = as_spread / 2
    mm_bid = reservation_price - half_spread
    mm_ask = reservation_price + half_spread

    return mm_bid, mm_ask, as_spread, skew

  
#unit test
# print("no inventory")
# print(compute_as_quotes(59999.95, 60000.05, 60000, 0.000079, 19, 0, 0.1))

# print("with inventory")
# print(compute_as_quotes(59999.95, 60000.05, 60000, 0.000079, 19, 2, 0.1))

In [11]:
def run_backtest(
    bars: pd.DataFrame,
    alpha: float = 0.5,
    N: float = 20,
    M: float = 10,
    sigma_window: int = 300,
) -> pd.DataFrame:

    cash = 0.0
    inventory = 0.0
    records = []

    log_ret = np.log(bars['mid']).diff().fillna(0).values
    sigma_arr = pd.Series(log_ret).rolling(sigma_window, min_periods=1).std().values

    for i, row in bars.iterrows():
        mid      = row['mid']
        best_bid = row['best_bid']
        best_ask = row['best_ask']
        spread   = best_ask - best_bid
        sigma    = sigma_arr[i] if sigma_arr[i] > 0 else 1e-6

        # Dynamic parameters
        fill_size     = alpha * spread / (sigma * mid)
        max_inventory = N * fill_size
        gamma         = (spread / 2) / (M * fill_size * sigma * mid)

        mm_bid, mm_ask, as_spread, skew = compute_as_quotes(
            best_bid, best_ask, mid, sigma, inventory, gamma
        )

        if row['any_sell'] and mm_bid >= row['min_sell_price'] and abs(inventory) < max_inventory:
            cash -= mm_bid * fill_size
            inventory += fill_size

        if row['any_buy'] and mm_ask <= row['max_buy_price'] and abs(inventory) < max_inventory:
            cash += mm_bid * fill_size
            inventory -= fill_size

        mtm = cash + inventory * mid
        records.append({
            'mid': mid, 'mm_bid': mm_bid, 'mm_ask': mm_ask,
            'skew': skew, 'inventory': inventory, 'cash': cash,
            'mtm_pnl': mtm, 'sigma': sigma, 'fill_size': fill_size,
            'max_inventory': max_inventory, 'gamma': gamma,
            'timestamp': row['timestamp'],
        })

    return pd.DataFrame(records)

In [12]:
def build_bars(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate tick-level trades into 1-second bars.
    """
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
    df = df.set_index('datetime')
    
    buys = df[df['sign'] == 1]
    sells = df[df['sign'] == -1]
    
    bars = pd.DataFrame({
        'mid'           : df['mid'].resample('1s').last(),
        'best_bid'      : df['best_bid'].resample('1s').last(),
        'best_ask'      : df['best_ask'].resample('1s').last(),
        'bar_trades'    : df['qty'].resample('1s').count(),
        'any_buy'       : buys['qty'].resample('1s').count() > 0,
        'any_sell'      : sells['qty'].resample('1s').count() > 0,
        'max_buy_price' : buys['price'].resample('1s').max(),
        'min_sell_price': sells['price'].resample('1s').min(),
        'buy_volume'    : buys['qty'].resample('1s').sum(),
        'sell_volume'   : sells['qty'].resample('1s').sum(),
        'toxic_rate'    : df['toxic'].resample('1s').mean(),
        'any_toxic' : df['toxic'].astype(int).resample('1s').max().astype(bool),
        'buy_trades'    : buys['qty'].resample('1s').count(),
        'sell_trades'   : sells['qty'].resample('1s').count(),
    })
    
    # Bars with no trades on one side get NaN — fill correctly
    bars['any_buy']  = bars['any_buy'].fillna(False)
    bars['any_sell'] = bars['any_sell'].fillna(False)
    bars['max_buy_price']  = bars['max_buy_price'].fillna(0)
    bars['min_sell_price'] = bars['min_sell_price'].fillna(0)
    bars['buy_volume']  = bars['buy_volume'].fillna(0)
    bars['sell_volume'] = bars['sell_volume'].fillna(0)
    bars['toxic_rate']  = bars['toxic_rate'].fillna(0)
    bars['buy_trades']  = bars['buy_trades'].fillna(0)
    bars['sell_trades'] = bars['sell_trades'].fillna(0)

    # Drop bars with no mid (empty bars)
    bars = bars.dropna(subset=['mid']).reset_index().rename(columns={'datetime': 'timestamp'})
    
    return bars

In [13]:
def summary_stats(results: pd.DataFrame) -> dict:
    # Resample mtm_pnl to daily, compute daily returns
    results.index = pd.to_datetime(results['timestamp'])
    daily_pnl = results['mtm_pnl'].resample('1D').last().diff().dropna()

    mean = daily_pnl.mean()
    std  = daily_pnl.std()
    sharpe = (mean / std * np.sqrt(365)) if std > 0 else 0
    
    

    cummax = results['mtm_pnl'].cummax()
    max_dd = (results['mtm_pnl'] - cummax).min()

    inv_95 = np.percentile(results['inventory'].abs(), 95)
    fill_rate = (results['inventory'].diff().abs() > 0).mean()

    return {
        'sharpe'          : round(sharpe, 3),
        'max_drawdown'    : round(max_dd, 4),
        'final_mtm'       : round(results['mtm_pnl'].iloc[-1], 4),
        'inventory_95pct' : round(inv_95, 4),
        'fill_rate'       : round(fill_rate, 4),
        'mean_bar_pnl'    : round(mean, 6),
        'std_bar_pnl'     : round(std, 6),
    }

In [14]:
#Setup & Data Loading
from pathlib import Path
import pandas as pd

data_dir = Path("../data/processed/features")

BACKTEST_COLS = [
    'timestamp', 'price', 'sign', 'qty',
    'best_bid', 'best_ask', 'midprice', 'toxic'
]

ASSETS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
WEEKS = ['week1', 'week2', 'week3']

BACKTEST_COLS = [
    'timestamp', 'price', 'sign', 'qty',
    'spread', 'midprice', 'toxic'
]

def load_backtest_data(data_dir, asset, week):
    path = Path(data_dir) / f"{asset}_{week}_full_features.parquet"
    df = pd.read_parquet(path, columns=BACKTEST_COLS)
    
    df = df.sort_values('timestamp').reset_index(drop=True)
    df = df.dropna()
    df['timestamp'] = df['timestamp'].astype('int64') / 1e9
    
    # Reconstruct best bid/ask
    df['best_bid'] = df['midprice'] - df['spread'] / 2
    df['best_ask'] = df['midprice'] + df['spread'] / 2
    df['mid'] = df['midprice']  # alias for backtest engine
    
    return df

# Load as dict, not concatenated DataFrame
all_data = {}
for asset in ASSETS:
    for week in WEEKS:
        all_data[(asset, week)] = load_backtest_data(data_dir, asset, week)
        print(f"  {asset} {week}: {len(all_data[(asset, week)]):,} trades")

  BTCUSDT week1: 10,071,946 trades
  BTCUSDT week2: 12,032,260 trades
  BTCUSDT week3: 24,249,845 trades
  ETHUSDT week1: 4,554,411 trades
  ETHUSDT week2: 7,137,925 trades
  ETHUSDT week3: 10,051,489 trades
  SOLUSDT week1: 3,968,322 trades
  SOLUSDT week2: 5,757,031 trades
  SOLUSDT week3: 11,434,844 trades


In [15]:
# Build bars for all assets/weeks
all_bars = {}
for asset in ASSETS:
    for week in WEEKS:
        bars = build_bars(all_data[(asset, week)])
        all_bars[(asset, week)] = bars
        print(f"{asset} {week}: {len(all_data[(asset, week)]):,} ticks → {len(bars):,} bars "
              f"| avg {all_data[(asset, week)]['qty'].sum()/len(bars):.3f} BTC/bar "
              f"| toxic_rate mean {bars['toxic_rate'].mean():.3f}")

BTCUSDT week1: 10,071,946 ticks → 526,551 bars | avg 1.580 BTC/bar | toxic_rate mean 0.010
BTCUSDT week2: 12,032,260 ticks → 525,290 bars | avg 1.807 BTC/bar | toxic_rate mean 0.009
BTCUSDT week3: 24,249,845 ticks → 558,416 bars | avg 1.915 BTC/bar | toxic_rate mean 0.042
ETHUSDT week1: 4,554,411 ticks → 379,955 bars | avg 12.980 BTC/bar | toxic_rate mean 0.018
ETHUSDT week2: 7,137,925 ticks → 415,317 bars | avg 15.472 BTC/bar | toxic_rate mean 0.020
ETHUSDT week3: 10,051,489 ticks → 319,539 bars | avg 25.369 BTC/bar | toxic_rate mean 0.080
SOLUSDT week1: 3,968,322 ticks → 409,128 bars | avg 134.348 BTC/bar | toxic_rate mean 0.025
SOLUSDT week2: 5,757,031 ticks → 428,987 bars | avg 138.642 BTC/bar | toxic_rate mean 0.031
SOLUSDT week3: 11,434,844 ticks → 460,117 bars | avg 232.029 BTC/bar | toxic_rate mean 0.126


In [16]:
import numpy as np
def compute_gamma(bars: pd.DataFrame, target_q_critical_fills: float = 5.0) -> float:
    """
    Set gamma so quotes go uncompetitive only after target_q_critical_fills 
    average fills worth of inventory.
    
    q_critical = half_spread / (gamma * sigma * mid)
    gamma = half_spread / (q_critical * sigma * mid)
    """
    sigma = np.log(bars['mid']).diff().std()
    mid = bars['mid'].mean()
    spread = (bars['best_ask'] - bars['best_bid']).mean()
    half_spread = spread / 2
    
    # avg fill size per bar
    avg_fill = (bars['buy_volume'] / bars['buy_trades'].replace(0, np.nan)).mean()
    q_critical = avg_fill * target_q_critical_fills
    
    gamma = half_spread / (q_critical * sigma * mid)
    return gamma

# Compute per asset
for asset in ASSETS:
    bars = all_bars[(asset, 'week1')]
    g = compute_gamma(bars)
    print(f"{asset}: gamma={g:.6f}")

BTCUSDT: gamma=0.037720
ETHUSDT: gamma=0.006668
SOLUSDT: gamma=0.007950


In [17]:
# Calibrate gamma on week1 for each asset
gammas = {}
for asset in ASSETS:
    gammas[asset] = compute_gamma(all_bars[(asset, 'week1')])
    print(f"{asset}: gamma={gammas[asset]:.6f}")

print()

# Run full table
results_all = {}
for asset in ASSETS:
    for week in WEEKS:
        bars = all_bars[(asset, week)]
        res = run_backtest(
            bars=bars,

        )
        stats = summary_stats(res)
        results_all[(asset, week)] = stats
        print(f"{asset} {week:5s} | Sharpe {stats['sharpe']:8.2f} | "
              f"MtM {stats['final_mtm']:10.2f} | "
              f"inv_95 {stats['inventory_95pct']:.3f} | "
              f"fill_rate {stats['fill_rate']:.3f}")

BTCUSDT: gamma=0.037720
ETHUSDT: gamma=0.006668
SOLUSDT: gamma=0.007950

BTCUSDT week1 | Sharpe    -5.09 | MtM   -1937.31 | inv_95 2.024 | fill_rate 0.436
BTCUSDT week2 | Sharpe   -17.19 | MtM   -6109.53 | inv_95 1.929 | fill_rate 0.400
BTCUSDT week3 | Sharpe    -0.54 | MtM    -989.02 | inv_95 1.270 | fill_rate 0.288
ETHUSDT week1 | Sharpe   -35.34 | MtM    -405.80 | inv_95 2.156 | fill_rate 0.515
ETHUSDT week2 | Sharpe    -9.71 | MtM    -478.48 | inv_95 5.138 | fill_rate 0.398
ETHUSDT week3 | Sharpe   -57.72 | MtM    -156.07 | inv_95 1.244 | fill_rate 0.418
SOLUSDT week1 | Sharpe  -167.67 | MtM    -369.56 | inv_95 0.573 | fill_rate 0.607
SOLUSDT week2 | Sharpe  -503.80 | MtM    -463.39 | inv_95 0.486 | fill_rate 0.620
SOLUSDT week3 | Sharpe   -12.79 | MtM    -260.64 | inv_95 33.096 | fill_rate 0.302


In [20]:
# Run all weeks per asset, concatenate results, compute Sharpe once
asset_results = {}
for asset in ASSETS:
    all_res = []
    for week in WEEKS:
        bars = all_bars[(asset, week)]
        res = run_backtest(bars=bars)
        all_res.append(res)
    combined = pd.concat(all_res).reset_index(drop=True)
    asset_results[asset] = combined

# Now compute Sharpe on 3 weeks of daily PnL (18-20 observations)
for asset in ASSETS:
    res = asset_results[asset]
    res.index = pd.to_datetime(res['timestamp'])
    daily = res['mtm_pnl'].resample('1D').last().diff().dropna()
    mean = daily.mean()
    std = daily.std()
    sharpe = mean / std * np.sqrt(365) if std > 0 else 0
    total_mtm = res['mtm_pnl'].iloc[-1]
    inv_95 = np.percentile(res['inventory'].abs(), 95)
    fill_rate = (res['inventory'].diff().abs() > 0).mean()
    print(f"{asset:10s} | Sharpe {sharpe:8.2f} | MtM {total_mtm:10.2f} | "
          f"inv_95 {inv_95:.3f} | fill_rate {fill_rate:.3f} | "
          f"daily_obs {len(daily)}")

KeyboardInterrupt: 